# Latent Studio — LoRA Training Pipeline

Automated **artist → downloaded → preprocessed → captioned → trained → validated → published LoRA** pipeline for the Latent Studio capstone. Generated from `training/*.py` (see `scripts/build_colab_notebook.py`), flattened top-to-bottom.

Produces the project's style LoRAs from public-domain (CC0) Art Institute of Chicago artworks and pushes the shipped ones to the Hugging Face Hub, from where the main app (`latent_studio_colab.ipynb`) loads them. **Six ship: Hokusai, Turner, Monet, Dürer, Hiroshige, Rembrandt.** The pipeline also trained Cézanne, Cassatt and Van Gogh — kept as documented limitations (dataset size was the ceiling, not model quality; see the roster table below) rather than pushed. This pipeline is a Colab workflow — deliberately **not** a live feature of the app.

**Runtime:** needs a **GPU** runtime (T4) for the training + validation steps. Run one artist per pass (see the last section).

## 1. Install dependencies

In [ ]:
#@title Install dependencies
%pip install -q "torch>=2.2" "torchvision>=0.17" "transformers>=4.41" "safetensors>=0.4" "requests>=2.31" "pillow>=10.0" "numpy>=1.26" "huggingface_hub>=0.34"

## 2. Hugging Face setup (required)
This notebook **pushes** the trained LoRAs to your account (one repo per artist, `espressosession/latent-studio-<artist>-lora`, created on first push), so the token here **must have `write` access** — a read token will create the repo and then fail on the upload, leaving an empty repo behind. It also lifts the anonymous rate limit on the multi-GB downloads.

The cell also **switches off Xet**, `huggingface_hub`'s default download backend, which hangs part-way into a large file on Colab (see the note in the source). Both settings are read when `huggingface_hub` is first imported, so **this cell must run before the imports below** — setting them later does nothing. If you have already imported anything, restart the runtime.

On Colab, store the token once as a secret named **`HF_TOKEN`** (🔑 icon in the left sidebar, 'Notebook access' on). Otherwise this falls back to an interactive login.

In [ ]:
#@title Hugging Face login
import os

# Read by huggingface_hub at import, so this must run before anything imports it.
# Xet is the default download backend and it hangs in Colab (see the note above).
os.environ["HF_HUB_DISABLE_XET"] = "1"


def _hf_token():
    if os.environ.get("HF_TOKEN"):
        return "HF_TOKEN env var"
    try:  # Colab secret (key icon in the sidebar, 'Notebook access' ON)
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
            return "Colab secret"
    except Exception:
        pass  # not on Colab, or no secret set / not shared with this notebook
    from huggingface_hub import get_token

    return "cached login" if get_token() else None


source = _hf_token()
if source is None:
    print(
        "No token found — anonymous downloads are rate-limited, so logging in.\n"
        "Tip: save it once as a Colab secret named HF_TOKEN (key icon, left sidebar)\n"
        "and this cell will pick it up silently from now on."
    )
    from huggingface_hub import login

    login()
else:
    print(f"Hugging Face token found ({source}) — downloads are authenticated and fast.")

## 3. Clone the diffusers training script

In [ ]:
#@title Clone diffusers
![ -d diffusers ] || git clone --depth 1 https://github.com/huggingface/diffusers.git
%pip install -q ./diffusers
%pip install -q -r diffusers/examples/text_to_image/requirements.txt
%pip install -q "accelerate>=0.31" "transformers>=4.41" "peft>=0.11" "datasets>=2.19" "bitsandbytes>=0.43" safetensors xformers ftfy
# Colab ships an old torchao (0.10) that current peft rejects during LoRA
# injection (`Found an incompatible version of torchao`); we don't use it.
%pip uninstall -q -y torchao

## 4. Imports and Globals

In [ ]:
#@title Imports
import os
import random
import time
import requests
import glob
import numpy as np
import json
import re
import argparse
import subprocess
import sys
import torch
import shutil
from dataclasses import dataclass
from PIL import Image, ImageDraw
from diffusers import StableDiffusionPipeline
from transformers import CLIPModel, CLIPProcessor


## 5. Training modules (reference)
One card per file in `training/`, in dependency order (`pipeline.py` references every other module, so it comes last) — run them all to define the pipeline below. Skip ahead if you're only here to run it.

### Config
_(`training/config.py`)_

Central config: the artist roster with their Art Institute of Chicago search terms, each one's **on-domain validation prompt** (a subject that artist actually worked on — one shared prompt would judge Rembrandt on a mountain landscape), the shared neutral `sks` trigger, the **`HF_USERNAME`** the LoRAs are pushed to, and every hyperparameter (dataset size, LoRA rank, steps-per-image, validation + scorecard settings). **Tweak here** to change artists or training settings — nothing downstream hard-codes them.

**Why the roster looks the way it does.** Every artist is public domain (CC0) via the AIC API, but AIC's public-domain *coverage* per artist varies enormously — and that turned out to be the real constraint on this pipeline, not model quality. Measured image yield, and the outcome it led to:

| artist | style | AIC public-domain images | outcome |
|---|---|---|---|
| Hokusai | ukiyo-e woodblock | plenty (trained 150) | **SHIP** |
| Turner | atmospheric oil landscape | plenty (trained 150) | **SHIP** |
| Dürer | Renaissance engraving/woodcut | plenty (trained 150) | **SHIP** (biggest gain from +data) |
| Hiroshige | ukiyo-e landscape | ~2270 (trained 150) | **SHIP** |
| Rembrandt | etching, chiaroscuro | ~980 (trained 150) | **SHIP** |
| Monet | Impressionist landscape | 40 (ceiling) | **SHIP** (on-domain only, exception) |
| Cézanne | Post-Impressionist | 32 (ceiling) | limitation — style never holds |
| Cassatt | Impressionist figures | 46 (ceiling; ~236 exist, most lost in preprocessing) | limitation — data-thin |
| Van Gogh | Post-Impressionist impasto | 19 (ceiling) | limitation — thin, and the corpus skews to B&W drawings |
| Toulouse-Lautrec | Art Nouveau poster | ~2600 | untrained |
| Renoir | Impressionist figures | ~780 | untrained |
| Daumier | lithograph caricature | ~630 | untrained |
| Utamaro | ukiyo-e portraits | ~280 | untrained |
| Goya | dark expressive prints | ~275 | untrained |
| Cameron | Victorian photography | ~150 | untrained (the only other woman with enough) |
| Kollwitz | Expressionist prints | 0 | impossible — d. 1945, still in copyright |

The public domain skews heavily male: of the women checked, only Cassatt and Cameron have enough public-domain work to train on (Morisot 20, Kauffmann 9, Bonheur 6) — a property of the commons, not of the shortlist.

All nine trained artists were (re)trained under the current pipeline (150-image target where the public domain allows, steps = images × 40) and scored on CUDA in a before/after study. Dataset size is the ceiling: the five that reached 150 images ship; Monet (40) ships as an on-domain-only exception; Cézanne/Cassatt/Van Gogh stay documented limitations. `registry.LORAS` is the shipped cut of six.

**`artist_match`** is the lowercased substring the returned `artist_title` must contain, and it must match AIC's *stored* spelling exactly (not necessarily the search spelling) — it's the guard against a same-surname artist slipping in. Notably: Dürer keeps the umlaut while Cézanne drops the accent in AIC's records; Daumier and Goya are stored under their full honorifics but the surname substring still matches; Cameron and Rembrandt need their *full* names, since AIC also holds unrelated artists sharing just the surname (David Young Cameron, Rembrandt Peale) whose work would otherwise blend into the style.

In [ ]:
#@title config.py
"""Config for the LoRA automation pipeline (artist name -> downloaded ->
preprocessed -> captioned -> trained -> validated -> pushed .safetensors).

Single source of truth shared by the pipeline modules and — for trigger words
and HF repo ids — by the app's registry.py, so what the app loads matches what
the pipeline produced. Tweak the ARTISTS table / hyperparameters here; nothing
downstream hard-codes an artist.
"""

# Hugging Face account the trained LoRAs are pushed to (and that the app's
# registry.py points at). The two repos are created on first push.
HF_USERNAME = "espressosession"

# Neutral, semantically-empty trigger shared by every project LoRA — the base
# model carries ~no prior for it, so any style at weight 0 is fully attributable
# to training. Keep in sync with registry.py.
TRIGGER_WORD = "sks"

@dataclass(frozen=True)
class Artist:
    id: str  # short slug, used in filenames + the HF repo name
    label: str  # human-readable, shown in the app's LoRA dropdown
    search_name: str  # queried against the Art Institute of Chicago `artist_title`
    artist_match: str  # lowercased substring the returned `artist_title` must contain
    trigger_word: str = TRIGGER_WORD  # neutral shared trigger; see TRIGGER_WORD above
    # A subject THIS artist actually worked on (content words only, no medium —
    # that's the trigger word's job). Falls back to VALIDATION_PROMPT.
    prompt: str = ""

    @property
    def hf_repo(self) -> str:
        return f"{HF_USERNAME}/latent-studio-{self.id}-lora"

    @property
    def validation_prompt(self) -> str:
        return self.prompt or VALIDATION_PROMPT

# The training roster — every artist is public domain (CC0) via the AIC API.
# AIC's public-domain *coverage* per artist is the real constraint on dataset size
# (see the notebook section for the measured yield table and the shipped/limitation
# split). Only TRAINED artists get a registry.LORAS entry. `artist_match` is the
# lowercased substring the returned `artist_title` must contain — it must match
# AIC's stored spelling exactly, and for a shared surname (Cameron, Rembrandt) needs
# the full name to avoid blending in a different artist's work.
ARTISTS: dict[str, Artist] = {
    "hokusai": Artist(
        id="hokusai",
        label="Hokusai (Ukiyo-e)",
        search_name="Katsushika Hokusai",
        artist_match="hokusai",
        prompt="a small boat crossing a river beneath a mountain, birds flying overhead",
    ),
    "turner": Artist(
        id="turner",
        label="Turner (Oil)",
        search_name="Joseph Mallord William Turner",
        artist_match="joseph mallord william turner",
        prompt="a sailing ship on a rough sea, the sun breaking through heavy haze",
    ),
    "cassatt": Artist(
        id="cassatt",
        label="Cassatt (Impressionism)",
        search_name="Mary Cassatt",
        artist_match="cassatt",
        prompt="a mother holding a small child beside a window",
    ),
    "vangogh": Artist(
        id="vangogh",
        label="Van Gogh (Post-Impressionism)",
        search_name="Vincent van Gogh",
        artist_match="van gogh",
        prompt="a wheat field under a swirling sky, cypress trees at the edge",
    ),
    "kollwitz": Artist(
        id="kollwitz",
        label="Kollwitz (Expressionism)",
        search_name="Käthe Kollwitz",
        artist_match="kollwitz",
        prompt="a woman holding her child close, her head bowed",
    ),
    "monet": Artist(
        id="monet",
        label="Monet (Impressionism)",
        search_name="Claude Monet",
        artist_match="monet",
        prompt="a pond with water lilies and a footbridge, morning light",
    ),
    "cezanne": Artist(
        id="cezanne",
        label="Cézanne (Post-Impressionism)",
        search_name="Paul Cezanne",
        artist_match="cezanne",  # AIC spells it without the accent here
        prompt="apples and a jug on a draped table",
    ),
    "duerer": Artist(
        id="duerer",
        label="Dürer (Renaissance)",
        search_name="Albrecht Dürer",
        artist_match="dürer",  # AIC keeps the umlaut — the search handles it fine
        prompt="a bearded man in a wide hat holding a book",
    ),
    "lautrec": Artist(
        id="lautrec",
        label="Toulouse-Lautrec (Art Nouveau)",
        search_name="Henri de Toulouse-Lautrec",
        artist_match="toulouse-lautrec",
        prompt="a dancer on a stage under bright lights",
    ),
    "hiroshige": Artist(
        id="hiroshige",
        label="Hiroshige (Ukiyo-e)",
        search_name="Utagawa Hiroshige",
        artist_match="utagawa hiroshige",
        prompt="travellers on a coastal road in the rain, a village in the distance",
    ),
    "rembrandt": Artist(
        id="rembrandt",
        label="Rembrandt (Etching)",
        search_name="Rembrandt van Rijn",
        artist_match="rembrandt van rijn",  # full name: guards against Rembrandt Peale
        prompt="an old man's face lit from one side, deep shadow behind him",
    ),
    "renoir": Artist(
        id="renoir",
        label="Renoir (Impressionism)",
        search_name="Pierre-Auguste Renoir",
        artist_match="renoir",
        prompt="people at a table in a sunlit garden",
    ),
    "daumier": Artist(
        id="daumier",
        label="Daumier (Lithograph)",
        search_name="Honoré Daumier",
        artist_match="daumier",  # AIC titles him "Honoré-Victorin Daumier"
        prompt="two men in coats and tall hats arguing in the street",
    ),
    "utamaro": Artist(
        id="utamaro",
        label="Utamaro (Ukiyo-e)",
        search_name="Kitagawa Utamaro",
        artist_match="utamaro",
        prompt="a woman with an elaborate hairstyle holding a fan",
    ),
    "goya": Artist(
        id="goya",
        label="Goya (Etching)",
        search_name="Francisco Goya",
        artist_match="goya",  # AIC titles him "Francisco José de Goya y Lucientes"
        prompt="a crowd of figures in the dark, one of them holding a lantern",
    ),
    "cameron": Artist(
        id="cameron",
        label="Cameron (Photography)",
        search_name="Julia Margaret Cameron",
        # MUST be the full name: AIC also holds David Young Cameron (a different
        # artist, ~18 works). A bare "cameron" would blend his etchings into her style.
        artist_match="julia margaret cameron",
        prompt="a woman with long loose hair, her head turned toward the light",
    ),
}

# Base checkpoint the LoRAs train on and are validated against.
BASE_MODEL = "stable-diffusion-v1-5/stable-diffusion-v1-5"

# --- dataset targets (step 1-2) ----------------------------------------------
TARGET_IMAGE_COUNT = 150  # stop once this many usable images survive preprocessing
DOWNLOAD_WIDTH = 1686  # IIIF request width — 2x margin so border-trimmed plates stay above MIN_IMAGE_SIZE
MIN_IMAGE_SIZE = 384  # discard images whose shorter side is smaller than this
MAX_ASPECT_RATIO = 1.7  # discard images more elongated than this (long/short side)
RESOLUTION = 512  # SD 1.5 training resolution

# --- training hyperparameters (step 4) ---------------------------------------
RANK = 24  # LoRA rank — higher captures more style detail (8-16 is a minimum for a style)
LEARNING_RATE = 1e-4
TRAIN_SEED = 1337

# Training length scales with the dataset (steps per image, not a fixed total) —
# see pipeline._training_steps(). A fixed total overtrains a small set and
# undertrains a large one.
STEPS_PER_IMAGE = 40
MIN_TRAIN_STEPS = 800  # floor, so a small set still converges
MAX_TRAIN_STEPS = 6000  # cap, so a huge set stays inside a sane Colab runtime
CHECKPOINT_COUNT = 8  # intermediate snapshots for the validation sweep -> steps // 8

# --- validation (step 5) -----------------------------------------------------
# Fallback on-domain prompt for artists that don't set their own `prompt` (see
# Artist.validation_prompt). Prefer the per-artist one: it is what makes the
# on-domain grids actually on-domain.
VALIDATION_PROMPT = "a small boat crossing a river beneath a mountain, birds flying overhead"
VALIDATION_SEEDS = [1000, 1001, 1002]
VALIDATION_CFG = 7.5
VALIDATION_STEPS = 30
# Swept at fixed seeds to surface burn-in — spans the app's Style-strength range;
# 0.0 is the neutral baseline, >1.0 is where a style typically starts "cooking".
VALIDATION_WEIGHTS = [0.0, 0.5, 1.0, 1.5]
# Deliberately MODERN subjects no pre-1900 painter could have depicted — if the
# style still shows, that's generalisation, not a memorised training motif (also
# what makes scorecard.py's style_gain metric airtight).
VALIDATION_OFFDOMAIN_PROMPTS = [
    "a cat sitting on a windowsill",
    "a busy city street with cars and traffic lights",
    "a plate of food on a kitchen table",
    "a person riding a bicycle",
]

# --- scorecard (step 5b) -----------------------------------------------------
# The scorecard answers the one question validation never asks: does the LoRA look
# like the ARTIST'S ACTUAL WORK? It compares generations against the real corpus
# instead of only against a no-LoRA baseline. See scorecard.py.
SCORECARD_CLIP_MODEL = "openai/clip-vit-base-patch32"
SCORECARD_REFERENCE_COUNT = 24  # real artworks embedded to form the style centroid
SCORECARD_WEIGHTS = [0.5, 1.0]  # LoRA strengths scored (0.0 is the implicit baseline)
SCORECARD_SEEDS = [1000, 1001]  # two seeds per prompt, so a metric isn't one lucky draw
# Thresholds are heuristics calibrated by eye, not laws — they sort LoRAs into
# "look at this one first", they don't ship anything on their own. The contact
# sheet is still the arbiter.
SCORECARD_MIN_STYLE_GAIN = 0.02  # below this, the adapter barely registers as a style
SCORECARD_MAX_PROMPT_DROP = 0.03  # above this, the LoRA stopped following the prompt
SCORECARD_MAX_BURN_IN = 1.25  # saturation ratio vs. baseline above which it's cooked


### Step 1 — download (Art Institute of Chicago)
_(`training/download.py`)_

Downloads public-domain (CC0) artworks from the Art Institute of Chicago Open Access API (no API key). Unlike the Met's full-text search, AIC lets us query the `artist_title` field directly and filter to `is_public_domain` — we additionally verify each result's artist name (`artist_match`) so no loose match (e.g. a different 'Turner') slips into the dataset.

In [ ]:
#@title download.py
"""Step 1 of the LoRA pipeline: download an artist's public-domain (CC0) artworks
from the Art Institute of Chicago Open Access API (no key). AIC over the Met for its
real `artist_title` filter + `is_public_domain` flag (the Met's full-text search left
only ~2 genuinely-Hokusai PD images); each result's name is verified against
`artist_match` so a loose match never slips in."""

SEARCH_URL = "https://api.artic.edu/api/v1/artworks/search"
IIIF_BASE = "https://www.artic.edu/iiif/2"
# AIC asks API users to identify themselves via this header.
HEADERS = {"AIC-User-Agent": "Latent Studio capstone (github.com/espressosession/latent-studio)"}

def _search_page(search_name: str, page: int, per_page: int) -> list[dict]:
    params = {
        "query[bool][must][][match][artist_title]": search_name,
        "query[bool][must][][term][is_public_domain]": "true",
        "fields": "id,artist_title,image_id",
        "limit": per_page,
        "page": page,
    }
    resp = requests.get(SEARCH_URL, params=params, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return resp.json().get("data", [])

def download_artist(
    search_name: str,
    out_dir: str,
    artist_match: str,
    target: int = 60,
    per_page: int = 100,
    request_delay: float = 0.1,
    pool_factor: int = 4,
    seed: int = 0,
    image_width: int = 1686,
) -> int:
    """Downloads up to `target` public-domain artworks whose `artist_title` contains
    `artist_match` into `out_dir`. Gathers a larger candidate pool first, then takes
    a deterministically shuffled subset so files spread across the artist's whole
    collection rather than clustering on the first few series. Returns the count
    downloaded."""
    os.makedirs(out_dir, exist_ok=True)

    candidates: list[dict] = []
    pool_cap = target * pool_factor
    page = 1
    while len(candidates) < pool_cap:
        try:
            results = _search_page(search_name, page, per_page)
        except Exception as exc:
            print(f"AIC search failed on page {page}: {exc}")
            break
        if not results:
            break
        for art in results:
            if artist_match not in (art.get("artist_title") or "").lower():
                continue  # loose match for a different artist — skip
            if art.get("image_id"):
                candidates.append(art)
        page += 1

    random.Random(seed).shuffle(candidates)  # spread across the collection, reproducibly
    selected = candidates[:target]

    saved = 0
    for i, art in enumerate(selected):
        url = f"{IIIF_BASE}/{art['image_id']}/full/{image_width},/0/default.jpg"
        try:
            content = requests.get(url, headers=HEADERS, timeout=60).content
        except Exception:
            continue
        with open(os.path.join(out_dir, f"{i:04d}_{art['id']}.jpg"), "wb") as f:
            f.write(content)
        saved += 1
        time.sleep(request_delay)  # be polite to the API

    print(f"'{search_name}': downloaded {saved} public-domain images (from {len(candidates)} candidates) -> {out_dir}")
    return saved


### Step 2 — preprocess
_(`training/preprocess.py`)_

Cleans the raw downloads into a training set: convert to RGB, center-crop to a square, resize to 512², and drop outliers (too small, or too elongated to crop sensibly). Targets ~25-30 usable images per artist.

In [ ]:
#@title preprocess.py
"""Step 2 of the LoRA pipeline: clean the raw downloads into a 512² training set —
trim frame/mount borders, RGB, center-crop, drop outliers — and report a colour
diagnostic (colourfulness + mono/sepia cast) so a tinted corpus is flagged, not
silently trained on. Writes sequentially numbered PNGs for the caption step."""

RAW_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff")

def center_crop_square(image: Image.Image) -> Image.Image:
    width, height = image.size
    side = min(width, height)
    left = (width - side) // 2
    top = (height - side) // 2
    return image.crop((left, top, left + side, top + side))

def trim_border(
    image: Image.Image, tol: int = 32, std_max: float = 20.0, light_min: int = 160, max_trim_frac: float = 0.30
) -> Image.Image:
    """Crop a uniform light paper mount / plate margin from the edges. A row/col
    counts as border if it's close to the corner-median background colour and
    near-uniform (robust to foxing specks/caption text); only fires on a light
    border, so a dark painted edge is never mistaken for a mount."""
    rgb = image.convert("RGB")
    arr = np.asarray(rgb).astype(np.float32)
    h, w, _ = arr.shape
    patch = max(2, min(h, w) // 40)
    corners = np.concatenate([
        arr[:patch, :patch].reshape(-1, 3), arr[:patch, -patch:].reshape(-1, 3),
        arr[-patch:, :patch].reshape(-1, 3), arr[-patch:, -patch:].reshape(-1, 3),
    ])
    bg = np.median(corners, axis=0)
    if bg.min() <= light_min:
        return rgb  # dark/coloured edge — a painting, not a paper mount

    row_border = (np.abs(np.median(arr, axis=1) - bg).max(axis=1) <= tol) & (arr.std(axis=1).mean(axis=1) <= std_max)
    col_border = (np.abs(np.median(arr, axis=0) - bg).max(axis=1) <= tol) & (arr.std(axis=0).mean(axis=1) <= std_max)

    def leading(flags: np.ndarray) -> int:
        i = 0
        while i < len(flags) and flags[i]:
            i += 1
        return i

    top = min(leading(row_border), int(h * max_trim_frac))
    bottom = h - min(leading(row_border[::-1]), int(h * max_trim_frac))
    left = min(leading(col_border), int(w * max_trim_frac))
    right = w - min(leading(col_border[::-1]), int(w * max_trim_frac))

    if bottom - top < h * 0.4 or right - left < w * 0.4:
        return rgb  # trimming ate too much — leave it untouched
    return rgb.crop((left, top, right, bottom))

def _opponent(rgb: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Red-green and yellow-blue opponent channels (Hasler-Süsstrunk)."""
    r, g, b = (rgb[..., i].astype(np.float32) for i in range(3))
    return r - g, 0.5 * (r + g) - b

def colourfulness(rgb: np.ndarray) -> float:
    """Hasler-Süsstrunk (2003) no-reference colourfulness. ~0-15 grayscale/
    monochrome, ~15-25 muted, 30+ vibrant. `rgb` is HxWx3 uint8."""
    rg, yb = _opponent(rgb)
    return float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean()))

def colour_cast(rgb: np.ndarray) -> tuple[float, float, bool]:
    """Detect a uniform colour cast (e.g. sepia). Returns
    (cast_strength, mono_ratio, warm): `mono_ratio` near 1 means most chroma points
    the same direction (a monochrome tint); `warm` True (red>green, yellowish) is
    the sepia/brown direction. `rgb` is HxWx3 uint8."""
    rg, yb = _opponent(rgb)
    cast = float(np.hypot(rg.mean(), yb.mean()))
    chroma = float(np.hypot(rg, yb).mean())
    mono = cast / (chroma + 1e-6)
    warm = bool(rg.mean() > 0 and yb.mean() > 0)
    return cast, mono, warm

def _report_colour(cf_scores: list[float], cast_scores: list[tuple[float, float, bool]]) -> None:
    cf = float(np.mean(cf_scores))
    cast = float(np.mean([c[0] for c in cast_scores]))
    mono = float(np.mean([c[1] for c in cast_scores]))
    warm_frac = float(np.mean([c[2] for c in cast_scores]))
    print(f"colour: colourfulness={cf:.1f}  cast={cast:.1f}  mono_ratio={mono:.2f}  warm={warm_frac:.0%}")
    # A sepia/tinted corpus reads as HIGH mono_ratio (most chroma one direction),
    # not low colourfulness — the tint itself can be quite saturated.
    if mono > 0.8 or cf < 15:
        if mono > 0.8 and warm_frac > 0.6:
            tone = "sepia/brown"
        elif mono > 0.8:
            tone = "single-hue"
        else:
            tone = "near-grayscale"
        print(
            f"  ! dataset looks monochrome/tinted ({tone}); the LoRA will inherit this cast. "
            "Consider a more colourful artist, or document it as a known limitation."
        )

def preprocess_folder(
    in_dir: str,
    out_dir: str,
    resolution: int = 512,
    min_size: int = 384,
    max_aspect: float = 1.7,
    target: int = 30,
) -> int:
    """Cleans images from `in_dir` into `out_dir`: trim frame borders, drop images
    whose shorter side is below `min_size` or whose long/short ratio exceeds
    `max_aspect`, then center-crop + resize. Stops once `target` images are
    written; prints a colour diagnostic. Returns the count kept."""
    os.makedirs(out_dir, exist_ok=True)
    paths = sorted(
        p for p in glob.glob(os.path.join(in_dir, "*")) if p.lower().endswith(RAW_EXTENSIONS)
    )

    kept = 0
    cf_scores: list[float] = []
    cast_scores: list[tuple[float, float, bool]] = []
    for path in paths:
        if kept >= target:
            break
        try:
            image = Image.open(path)
            image.load()
        except Exception:
            continue
        image = trim_border(image)  # remove frame/mount bands before measuring aspect
        width, height = image.size
        if min(width, height) < min_size:
            continue
        if max(width, height) / min(width, height) > max_aspect:
            continue
        clean = center_crop_square(image).resize((resolution, resolution), Image.LANCZOS)
        clean.save(os.path.join(out_dir, f"{kept:03d}.png"))
        arr = np.asarray(clean)
        cf_scores.append(colourfulness(arr))
        cast_scores.append(colour_cast(arr))
        kept += 1

    print(f"kept {kept}/{len(paths)} images -> {out_dir}")
    if kept:
        _report_colour(cf_scores, cast_scores)
    return kept


### Step 3 — caption (BLIP)
_(`training/captioning.py`)_

Captions each image with BLIP-large (a local model, no API key) and writes diffusers' `metadata.jsonl`. Each caption is `<trigger>, <content>`, and medium/style words (painting, drawing, engraving, sepia, …) are stripped from the content so those attributes attach only to the trigger word — that's what makes the trigger monopolise the *style* while the words carry the *content*.

In [ ]:
#@title captioning.py
"""Step 3 of the LoRA pipeline: caption each image with BLIP (local, no API key) and
write a diffusers `metadata.jsonl`. Each caption is `"<trigger>, <content>"` with
medium/style words (painting, drawing, engraving, sepia, …) stripped from the
content, so the trigger monopolises the *style* and the words carry only the
*content* — the decoupling a style LoRA needs."""

_processor = None
_model = None

# Words naming the medium/style rather than the content — removed from captions so
# they don't leak into content tokens (they should live on the trigger word only).
_MEDIUM_WORDS = [
    "black and white", "black-and-white", "oil painting", "watercolour", "watercolor",
    "woodblock print", "woodblock", "engraving", "etching", "mezzotint", "lithograph",
    "painting", "drawing", "sketch", "illustration", "print", "artwork", "photograph",
    "photo", "picture", "image", "poster", "sepia", "monochrome", "grayscale", "greyscale",
]
_MEDIUM_RE = re.compile(r"\b(" + "|".join(re.escape(w) for w in _MEDIUM_WORDS) + r")\b", re.IGNORECASE)

def _clean_caption(text: str) -> str:
    """Strip medium/style words and tidy the leftovers (BLIP's "a painting of X"
    becomes "a X"). Falls back to a neutral phrase if nothing content-y remains."""
    text = _MEDIUM_RE.sub(" ", text.lower())
    text = re.sub(r"\b(a|an|the)\s+of\b", r"\1", text)  # "a of mountain" -> "a mountain"
    text = re.sub(r"\b(a|an|the)\s+(a|an|the)\b", r"\2", text)  # collapse "a a" / "an a"
    text = re.sub(r"\s+", " ", text).strip(" ,.")
    text = re.sub(r"^(of|with|in|on)\b", "", text).strip(" ,.")
    return text or "a scene"

def _load_blip():
    global _processor, _model
    if _model is None:
        from transformers import BlipForConditionalGeneration, BlipProcessor

        name = "Salesforce/blip-image-captioning-large"
        _processor = BlipProcessor.from_pretrained(name)
        _model = BlipForConditionalGeneration.from_pretrained(name)
    return _processor, _model

def caption_folder(image_dir: str, trigger_word: str, max_new_tokens: int = 30) -> str:
    """Captions every PNG in `image_dir` and writes `metadata.jsonl` alongside
    them (the format diffusers' `--train_data_dir` + `--caption_column=text`
    expects). Returns the metadata path."""
    import torch
    from PIL import Image

    processor, model = _load_blip()
    paths = sorted(glob.glob(os.path.join(image_dir, "*.png")))
    if not paths:
        raise SystemExit(f"No images to caption in {image_dir}")

    meta_path = os.path.join(image_dir, "metadata.jsonl")
    with open(meta_path, "w", encoding="utf-8") as f:
        for path in paths:
            image = Image.open(path).convert("RGB")
            inputs = processor(image, return_tensors="pt")
            with torch.no_grad():
                output = model.generate(**inputs, max_new_tokens=max_new_tokens)
            description = processor.decode(output[0], skip_special_tokens=True).strip()
            caption = f"{trigger_word}, {_clean_caption(description)}"
            f.write(json.dumps({"file_name": os.path.basename(path), "text": caption}) + "\n")

    print(f"captioned {len(paths)} images -> {meta_path}")
    return meta_path


### Step 4 — train
_(`training/train_lora.py`)_

The training run itself: a thin wrapper around diffusers' official `train_text_to_image_lora.py` (LoRA via peft on the UNet, text encoder frozen, fp16). Needs the cloned diffusers checkout from the setup cell above; saves the final weights + intermediate `checkpoint-*` snapshots.

In [ ]:
#@title train_lora.py
"""Step 4 of the LoRA pipeline: the training run — a thin subprocess wrapper around
diffusers' examples/text_to_image/train_text_to_image_lora.py (peft on the UNet, text
encoder frozen). Needs a cloned diffusers checkout (the notebook clones it) and reads
captions from metadata.jsonl if captioning.py wrote one, else a trivial fallback.

    run_training(...)                                                          # from pipeline.py
    python training/train_lora.py --dataset_dir ... --output_dir ... --trigger_word ...   # CLI
"""

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")

def prepare_dataset(dataset_dir: str, trigger_word: str) -> str:
    """Fallback metadata.jsonl builder — only used if captioning.py hasn't
    already written one. Caption per image: a sibling <name>.txt if present,
    else just the trigger word."""
    meta_path = os.path.join(dataset_dir, "metadata.jsonl")
    if os.path.exists(meta_path):
        print(f"Using existing captions at {meta_path}")
        return meta_path

    images = [
        f for f in glob.glob(os.path.join(dataset_dir, "*")) if f.lower().endswith(IMAGE_EXTENSIONS)
    ]
    if not images:
        raise SystemExit(f"No images found in {dataset_dir}")

    with open(meta_path, "w", encoding="utf-8") as f:
        for image_path in images:
            caption_path = os.path.splitext(image_path)[0] + ".txt"
            if os.path.exists(caption_path):
                caption = open(caption_path, encoding="utf-8").read().strip()
            else:
                caption = trigger_word
            f.write(json.dumps({"file_name": os.path.basename(image_path), "text": caption}) + "\n")

    print(f"{len(images)} images -> {meta_path} (fallback captions)")
    return meta_path

def _ensure_diffusers_repo(diffusers_repo: str) -> str:
    """Ensure the diffusers example training script is available locally and
    return its path — clones the release tag matching the installed diffusers
    version (falls back to `main` for a dev/source install)."""
    script = os.path.join(diffusers_repo, "examples", "text_to_image", "train_text_to_image_lora.py")
    if os.path.exists(script):
        return script

    import diffusers

    version = diffusers.__version__.split("+")[0]
    if ".dev" in version:
        clone = ["git", "clone", "--depth", "1", "https://github.com/huggingface/diffusers.git", diffusers_repo]
        print(f"Cloning diffusers main (installed {version} is a dev build) for the training script...")
    else:
        clone = ["git", "clone", "--depth", "1", "--branch", f"v{version}",
                 "https://github.com/huggingface/diffusers.git", diffusers_repo]
        print(f"Cloning diffusers v{version} (matching the installed version) for the training script...")
    subprocess.run(clone, check=True)
    if not os.path.exists(script):
        raise SystemExit(f"Cloned diffusers but the training script is still missing at {script}")
    return script

def run_training(
    dataset_dir: str,
    output_dir: str,
    trigger_word: str,
    model_name: str = "stable-diffusion-v1-5/stable-diffusion-v1-5",
    diffusers_repo: str = "diffusers",
    max_train_steps: int = 1200,
    learning_rate: float = 1e-4,
    rank: int = 16,
    resolution: int = 512,
    train_batch_size: int = 1,
    gradient_accumulation_steps: int = 1,
    seed: int = 1337,
    checkpointing_steps: int = 300,
    resume_from_checkpoint: str | None = None,
) -> str:
    """Runs LoRA training and returns `output_dir` (final weights + checkpoint-*).
    `resume_from_checkpoint="latest"` continues an interrupted run instead of
    restarting from step 0 — only meaningful when `output_dir` persists (Drive)."""
    prepare_dataset(dataset_dir, trigger_word)

    import torch
    from accelerate.utils import write_basic_config

    # 8-bit Adam (bitsandbytes) and fp16 mixed precision both need CUDA; on any
    # other device fall back so the invocation stays valid instead of crashing on
    # a bitsandbytes/CUDA error. Real training still wants a CUDA GPU — this
    # pipeline is meant to run in Colab (T4), not locally on a Mac.
    on_cuda = torch.cuda.is_available()
    precision = "fp16" if on_cuda else "no"
    write_basic_config(mixed_precision=precision)

    # Let unsupported ops fall back to CPU when training on Apple Silicon (MPS).
    if not on_cuda and torch.backends.mps.is_available():
        os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

    training_script = _ensure_diffusers_repo(diffusers_repo)

    os.makedirs(output_dir, exist_ok=True)

    cmd = [
        sys.executable, "-m", "accelerate.commands.launch",
        f"--mixed_precision={precision}",
        training_script,
        f"--pretrained_model_name_or_path={model_name}",
        f"--train_data_dir={dataset_dir}",
        "--caption_column=text",
        f"--resolution={resolution}",
        "--center_crop",
        "--random_flip",
        f"--train_batch_size={train_batch_size}",
        f"--gradient_accumulation_steps={gradient_accumulation_steps}",
        "--gradient_checkpointing",
        *(["--use_8bit_adam"] if on_cuda else []),
        f"--mixed_precision={precision}",
        f"--max_train_steps={max_train_steps}",
        f"--learning_rate={learning_rate}",
        "--lr_scheduler=cosine",
        "--lr_warmup_steps=0",
        f"--rank={rank}",
        f"--seed={seed}",
        f"--checkpointing_steps={checkpointing_steps}",
        *([f"--resume_from_checkpoint={resume_from_checkpoint}"] if resume_from_checkpoint else []),
        f"--validation_prompt={trigger_word}",
        "--validation_epochs=1",
        f"--output_dir={output_dir}",
        # DataLoader workers spawn subprocesses that must pickle the script's local
        # transform fn — fine on Colab (Linux/fork), but breaks on macOS/Python 3.13
        # (spawn: "Can't get local object 'preprocess_train'"). Use workers only on CUDA.
        f"--dataloader_num_workers={2 if on_cuda else 0}",
    ]
    print("Running:", " ".join(cmd))
    # Stream the child's output line-by-line into the notebook. subprocess.run()
    # would let the training script write to the OS-level stdout/stderr, which
    # Jupyter/Colab does NOT capture into the cell — so real errors would vanish
    # into the runtime log and only an opaque CalledProcessError would surface.
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(
            f"LoRA training failed (exit code {process.returncode}). The diffusers/accelerate "
            "error is in the streamed output just above this message."
        )

    lora_files = glob.glob(os.path.join(output_dir, "*.safetensors"))
    print("Trained LoRA weights:", lora_files)
    return output_dir

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset_dir", required=True, help="Folder of training images (+ metadata.jsonl / .txt captions)")
    parser.add_argument("--output_dir", required=True, help="Where the trained .safetensors LoRA is written")
    parser.add_argument("--trigger_word", required=True, help="Caption fallback + validation prompt, e.g. 'sks'")
    parser.add_argument("--model_name", default="stable-diffusion-v1-5/stable-diffusion-v1-5")
    parser.add_argument("--diffusers_repo", default="diffusers", help="Path to a cloned huggingface/diffusers checkout")
    parser.add_argument("--max_train_steps", type=int, default=1200)
    parser.add_argument("--learning_rate", type=float, default=1e-4)
    parser.add_argument("--rank", type=int, default=16)
    parser.add_argument("--resolution", type=int, default=512)
    parser.add_argument("--train_batch_size", type=int, default=1)
    parser.add_argument("--gradient_accumulation_steps", type=int, default=1)
    parser.add_argument("--seed", type=int, default=1337)
    parser.add_argument("--checkpointing_steps", type=int, default=300)
    parser.add_argument("--resume_from_checkpoint", default=None, help="'latest' or a checkpoint-N dir to continue an interrupted run")
    return parser.parse_args()


### Step 5 — validate (mandatory gate)
_(`training/validation.py`)_

Generates a checkpoint-comparison grid (a no-LoRA baseline plus each snapshot), a strength sweep to surface burn-in, and a style-generalisation grid across on- AND off-domain prompts (portrait, animal, city, still life) to check the style transfers beyond the training motifs — plus a saturation-based recommendation. The sweep + generalisation grids are built on the **recommended** snapshot (the one you'll export), not blindly on `final`. **A LoRA counts as done only once you have actually looked at these grids** — 'trained automatically' must not mean 'shipped unchecked'.

In [ ]:
#@title validation.py
"""Step 5 — the mandatory validation gate: a LoRA is "done" only once these grids
have been looked at (checkpoint comparison, weight sweep, on/off-domain
generalisation, plus a saturation-based recommended snapshot — see the
notebook's Step 5 section). Self-contained (no latent_studio import)."""

def _device_dtype() -> tuple[str, torch.dtype]:
    if torch.cuda.is_available():
        return "cuda", torch.float16
    if torch.backends.mps.is_available():
        return "mps", torch.float16
    return "cpu", torch.float32

def _generator(device: str, seed: int) -> torch.Generator:
    gen_device = "cpu" if device == "mps" else device  # mps generator is unreliable
    return torch.Generator(device=gen_device).manual_seed(seed)

def _grid(images: list[Image.Image], rows: int, cols: int, labels=None, cell: int = 256) -> Image.Image:
    grid = Image.new("RGB", (cols * cell, rows * cell), (28, 28, 28))
    draw = ImageDraw.Draw(grid)
    for i, image in enumerate(images):
        r, c = divmod(i, cols)
        grid.paste(image.resize((cell, cell)), (c * cell, r * cell))
        if labels and i < len(labels) and labels[i]:
            draw.rectangle([c * cell, r * cell, c * cell + cell, r * cell + 14], fill=(0, 0, 0))
            draw.text((c * cell + 4, r * cell + 3), labels[i], fill=(255, 255, 255))
    return grid

def _burn_in_score(image: Image.Image) -> float:
    """Mean HSV saturation in [0, 1]; overfit/cooked styles oversaturate."""
    hsv = np.asarray(image.convert("HSV"), dtype=np.float32)
    return float(hsv[..., 1].mean() / 255.0)

def _checkpoint_step(name: str) -> int:
    if name == "final":
        return 10**9  # sort final last (it's the most-trained)
    return int(name.rsplit("-", 1)[-1]) if name.startswith("checkpoint-") else 0

def find_lora_dirs(output_dir: str) -> list[tuple[str, str]]:
    """(label, dir) for every LoRA snapshot that has loadable weights: the final
    weights in output_dir, plus any checkpoint-* subdir that carries a
    .safetensors (depends on the diffusers version's checkpoint save hook)."""
    dirs: list[tuple[str, str]] = []
    if glob.glob(os.path.join(output_dir, "*.safetensors")):
        dirs.append(("final", output_dir))
    for ckpt in glob.glob(os.path.join(output_dir, "checkpoint-*")):
        if glob.glob(os.path.join(ckpt, "*.safetensors")):
            dirs.append((os.path.basename(ckpt), ckpt))
    return sorted(dirs, key=lambda pair: _checkpoint_step(pair[0]))

def _recommend(scores: dict[str, float]) -> str | None:
    """Latest snapshot whose saturation isn't an outlier vs. the calmest one."""
    if not scores:
        return None
    calmest = min(scores.values())
    healthy = [name for name, score in scores.items() if score <= calmest * 1.25]
    pool = healthy or list(scores)
    return max(pool, key=_checkpoint_step)

def validate(
    output_dir: str,
    base_model: str,
    trigger_word: str,
    content_prompt: str,
    seeds: list[int],
    cfg: float = 7.5,
    steps: int = 30,
    weights: tuple[float, ...] = (0.5, 0.75, 1.0),
    offdomain_prompts: tuple[str, ...] = (),
) -> dict:
    """Runs the validation gate. Returns a dict with `checkpoint_grid`,
    `weight_grid`, `generalization_grid`, per-snapshot `scores`, the `recommended`
    snapshot label, and the discovered `lora_dirs`."""
    device, dtype = _device_dtype()
    pipe = StableDiffusionPipeline.from_pretrained(
        base_model, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False
    ).to(device)

    styled_prompt = f"{trigger_word}, {content_prompt}"
    cols = len(seeds)

    def gen(prompt, seed, scale=None):
        kwargs = {"cross_attention_kwargs": {"scale": scale}} if scale is not None else {}
        return pipe(
            prompt=prompt, num_inference_steps=steps, guidance_scale=cfg,
            generator=_generator(device, seed), **kwargs,
        ).images[0]

    # --- grid 1: no-LoRA baseline + one row per snapshot at strength 1.0 ---
    baseline = [gen(content_prompt, s) for s in seeds]
    images = list(baseline)
    labels = [f"no-LoRA s={s}" for s in seeds]

    lora_dirs = find_lora_dirs(output_dir)
    scores: dict[str, float] = {}
    for name, path in lora_dirs:
        pipe.load_lora_weights(path)
        row = [gen(styled_prompt, s, scale=1.0) for s in seeds]
        pipe.unload_lora_weights()
        scores[name] = float(np.mean([_burn_in_score(im) for im in row]))
        images.extend(row)
        labels.extend(f"{name} s={s}" for s in seeds)

    checkpoint_grid = _grid(images, 1 + len(lora_dirs), cols, labels)

    # Pick the sweep snapshot before grids 2/3, so they judge the recommended
    # snapshot rather than `final`, which is often already past burn-in.
    recommended = _recommend(scores)
    sweep_label, sweep_path = None, None
    if lora_dirs:
        by_label = dict(lora_dirs)
        sweep_label = recommended if recommended in by_label else lora_dirs[-1][0]
        sweep_path = by_label[sweep_label]

    # --- grid 2: recommended snapshot across strengths (burn-in) ---
    weight_grid = None
    if sweep_path is not None:
        pipe.load_lora_weights(sweep_path)
        w_images, w_labels = [], []
        for w in weights:
            for s in seeds:
                w_images.append(gen(styled_prompt, s, scale=w))
                w_labels.append(f"w={w} s={s}")
        pipe.unload_lora_weights()
        weight_grid = _grid(w_images, len(weights), cols, w_labels)

    # --- grid 3: style generalisation across on/off-domain prompts (recommended) ---
    generalization_grid = None
    if sweep_path is not None and offdomain_prompts:
        pipe.load_lora_weights(sweep_path)
        gen_seeds = seeds[:2]  # keep the grid small — this is about subject, not seed spread
        prompts = [content_prompt, *offdomain_prompts]
        g_images, g_labels = [], []
        for prompt in prompts:
            for s in gen_seeds:
                g_images.append(gen(f"{trigger_word}, {prompt}", s, scale=1.0))
                g_labels.append(f"{prompt[:22]} s={s}")
        pipe.unload_lora_weights()
        generalization_grid = _grid(g_images, len(prompts), len(gen_seeds), g_labels)

    print("Burn-in scores (mean saturation, higher = more cooked):")
    for name, score in scores.items():
        print(f"  {name}: {score:.3f}")
    print(f"Recommended snapshot (CONFIRM VISUALLY): {recommended}")
    print(f"Weight + generalisation grids built on: {sweep_label}")

    return {
        "checkpoint_grid": checkpoint_grid,
        "weight_grid": weight_grid,
        "generalization_grid": generalization_grid,
        "scores": scores,
        "recommended": recommended,
        "sweep_snapshot": sweep_label,
        "lora_dirs": lora_dirs,
    }

def render_snapshot_grids(
    snapshot_dir: str,
    base_model: str,
    trigger_word: str,
    content_prompt: str,
    seeds: list[int],
    cfg: float = 7.5,
    steps: int = 30,
    weights: tuple[float, ...] = (0.0, 0.5, 1.0, 1.5),
    offdomain_prompts: tuple[str, ...] = (),
) -> dict:
    """Re-render just the weight-sweep + generalisation grids for ONE snapshot dir
    — no checkpoint comparison, no retraining. Returns {"weight_grid",
    "generalization_grid"}."""
    device, dtype = _device_dtype()
    pipe = StableDiffusionPipeline.from_pretrained(
        base_model, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False
    ).to(device)

    def gen(prompt, seed, scale=None):
        kwargs = {"cross_attention_kwargs": {"scale": scale}} if scale is not None else {}
        return pipe(
            prompt=prompt, num_inference_steps=steps, guidance_scale=cfg,
            generator=_generator(device, seed), **kwargs,
        ).images[0]

    styled_prompt = f"{trigger_word}, {content_prompt}"
    cols = len(seeds)
    pipe.load_lora_weights(snapshot_dir)

    # weight sweep (weights x seeds) on the on-domain prompt
    w_images, w_labels = [], []
    for w in weights:
        for s in seeds:
            w_images.append(gen(styled_prompt, s, scale=w))
            w_labels.append(f"w={w} s={s}")
    weight_grid = _grid(w_images, len(weights), cols, w_labels)

    # generalisation across on/off-domain prompts (first 2 seeds) at strength 1.0
    generalization_grid = None
    if offdomain_prompts:
        gen_seeds = seeds[:2]
        prompts = [content_prompt, *offdomain_prompts]
        g_images, g_labels = [], []
        for prompt in prompts:
            for s in gen_seeds:
                g_images.append(gen(f"{trigger_word}, {prompt}", s, scale=1.0))
                g_labels.append(f"{prompt[:22]} s={s}")
        generalization_grid = _grid(g_images, len(prompts), len(gen_seeds), g_labels)

    pipe.unload_lora_weights()
    del pipe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {"weight_grid": weight_grid, "generalization_grid": generalization_grid}


### Step 5b — scorecard (does it look like the artist?)
_(`training/scorecard.py`)_

Step 5 only ever compares the LoRA against a *no-LoRA baseline* — it never looks at the artist's actual work, so 'does this resemble the source data?' stayed a gut call. This step measures it. Against the same prompt+seed rendered without the LoRA, it reports **style gain** (CLIP similarity to the real corpus, measured on deliberately *modern* prompts — a bicycle, traffic lights — that no public-domain corpus can contain, so the gain can only be style and never a memorised training motif), **prompt drop** (did the LoRA stop listening to the prompt? the tell-tale of overfitting) and **burn-in** (oversaturation). It also renders a contact sheet with the artist's REAL artworks in the top row — the numbers rank the LoRAs, your eye still decides. Works on a Hub repo id too, so you can grade what is actually *published*.

In [ ]:
#@title scorecard.py
"""Step 5b — the scorecard: does this LoRA look like the artist's real work?
Every metric is a delta vs. the same prompt+seed without the LoRA (style_gain,
prompt_drop, burn_in — see the notebook's Step 5b section). Takes a Hub repo id
or a local snapshot dir; self-contained (no latent_studio import)."""

def _device_dtype() -> tuple[str, torch.dtype]:
    if torch.cuda.is_available():
        return "cuda", torch.float16
    if torch.backends.mps.is_available():
        return "mps", torch.float16
    return "cpu", torch.float32

def _generator(device: str, seed: int) -> torch.Generator:
    gen_device = "cpu" if device == "mps" else device  # mps generator is unreliable
    return torch.Generator(device=gen_device).manual_seed(seed)

def _saturation(image: Image.Image) -> float:
    hsv = np.asarray(image.convert("HSV"), dtype=np.float32)
    return float(hsv[..., 1].mean() / 255.0)

# --- reference corpus ---------------------------------------------------------

def reference_images(artist_id: str, work_dir: str = "lora_work", count: int = SCORECARD_REFERENCE_COUNT) -> list[Image.Image]:
    """The artist's real artworks, to measure our generations against. Prefers the
    local training set; otherwise re-fetches + preprocesses a fresh CC0 sample from
    AIC, so a LoRA can be graded even when its training set lives elsewhere."""
    artist = ARTISTS[artist_id]
    dataset = os.path.join(work_dir, artist_id, "dataset")
    paths = sorted(glob.glob(os.path.join(dataset, "*.png")))[:count]

    if not paths:
        clean = os.path.join(work_dir, artist_id, "reference")
        paths = sorted(glob.glob(os.path.join(clean, "*.png")))[:count]
        if not paths:
            raw = os.path.join(work_dir, artist_id, "reference_raw")
            print(f"No local training set for '{artist_id}' — fetching a reference sample from AIC.")
            download_artist(
                artist.search_name, raw, artist.artist_match,
                target=count * 2, image_width=DOWNLOAD_WIDTH,
            )
            preprocess_folder(
                raw, clean, resolution=RESOLUTION, min_size=MIN_IMAGE_SIZE,
                max_aspect=MAX_ASPECT_RATIO, target=count,
            )
            paths = sorted(glob.glob(os.path.join(clean, "*.png")))[:count]

    if not paths:
        raise SystemExit(f"No reference images available for '{artist_id}'.")
    return [Image.open(p).convert("RGB") for p in paths]

# --- CLIP ---------------------------------------------------------------------

_clip_cache: dict[str, tuple] = {}

def _clip(model_name: str = SCORECARD_CLIP_MODEL):
    """CLIP, loaded once per process. fp32 on purpose — it is small, and the metrics
    are differences of cosines, where fp16 noise is the same size as the signal."""
    if model_name not in _clip_cache:
        device, _ = _device_dtype()
        model = CLIPModel.from_pretrained(model_name).to(device).eval()
        processor = CLIPProcessor.from_pretrained(model_name)
        _clip_cache[model_name] = (model, processor, device)
    return _clip_cache[model_name]

def _as_embedding(features) -> torch.Tensor:
    """transformers 4 returns the projected embedding as a plain tensor; transformers 5
    wraps it in a BaseModelOutputWithPooling (the embedding is `pooler_output`). Colab
    and this laptop are on different major versions, so accept both."""
    if isinstance(features, torch.Tensor):
        return features
    return features.pooler_output

def embed_images(images: list[Image.Image]) -> torch.Tensor:
    model, processor, device = _clip()
    with torch.no_grad():
        inputs = processor(images=images, return_tensors="pt").to(device)
        features = _as_embedding(model.get_image_features(**inputs))
    return torch.nn.functional.normalize(features, dim=-1).float().cpu()

def embed_texts(texts: list[str]) -> torch.Tensor:
    model, processor, device = _clip()
    with torch.no_grad():
        inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
        features = _as_embedding(model.get_text_features(**inputs))
    return torch.nn.functional.normalize(features, dim=-1).float().cpu()

def style_centroid(images: list[Image.Image]) -> torch.Tensor:
    """One vector standing for "what this artist's work looks like to CLIP"."""
    return torch.nn.functional.normalize(embed_images(images).mean(dim=0), dim=-1)

# --- contact sheet ------------------------------------------------------------

def _sheet(rows: list[tuple[str, list[Image.Image]]], cell: int = 224) -> Image.Image:
    """Rows of (label, images). Row 0 is the artist's real work; the rest are ours,
    one row per LoRA strength, columns aligned by prompt."""
    cols = max(len(images) for _, images in rows)
    label_h = 18
    sheet = Image.new("RGB", (cols * cell, len(rows) * (cell + label_h)), (24, 24, 24))
    draw = ImageDraw.Draw(sheet)
    for r, (label, images) in enumerate(rows):
        y = r * (cell + label_h)
        draw.text((4, y + 4), label, fill=(255, 255, 255))
        for c, image in enumerate(images[:cols]):
            sheet.paste(image.resize((cell, cell)), (c * cell, y + label_h))
    return sheet

# --- the scorecard itself -----------------------------------------------------

def score_lora(
    artist_id: str,
    lora_source: str | None = None,
    work_dir: str = "lora_work",
    weights: list[float] = SCORECARD_WEIGHTS,
    seeds: list[int] = SCORECARD_SEEDS,
    cfg: float = VALIDATION_CFG,
    steps: int = VALIDATION_STEPS,
    reference_count: int = SCORECARD_REFERENCE_COUNT,
) -> dict:
    """Grade one LoRA against the artist's real work. `lora_source` is a HF repo id
    or a local snapshot dir; None grades whatever is published (artist.hf_repo).
    Returns metrics per weight, a `recommended_weight`, `verdict`, and `sheet`."""
    artist = ARTISTS[artist_id]
    source = lora_source or artist.hf_repo
    device, dtype = _device_dtype()

    reference = reference_images(artist_id, work_dir, reference_count)
    centroid = style_centroid(reference)

    ondomain = artist.validation_prompt
    offdomain = list(VALIDATION_OFFDOMAIN_PROMPTS)
    prompts = [ondomain, *offdomain]

    pipe = StableDiffusionPipeline.from_pretrained(
        BASE_MODEL, torch_dtype=dtype, safety_checker=None, requires_safety_checker=False
    ).to(device)

    def render(prompt: str, seed: int, scale: float | None) -> Image.Image:
        # The trigger only belongs in the prompt when the adapter is loaded; the
        # baseline must be a *clean* base-model render of the same words.
        text = f"{artist.trigger_word}, {prompt}" if scale is not None else prompt
        kwargs = {"cross_attention_kwargs": {"scale": scale}} if scale is not None else {}
        return pipe(
            prompt=text, num_inference_steps=steps, guidance_scale=cfg,
            generator=_generator(device, seed), **kwargs,
        ).images[0]

    print(f"[{artist_id}] scoring {source} against {len(reference)} real artworks")

    # Baseline: no LoRA at all. Every metric below is measured relative to this.
    baseline = {(p, s): render(p, s, None) for p in prompts for s in seeds}

    pipe.load_lora_weights(source)
    styled = {
        (w, p, s): render(p, s, w)
        for w in weights for p in prompts for s in seeds
    }
    pipe.unload_lora_weights()

    def cosine_to_corpus(images: list[Image.Image]) -> float:
        return float((embed_images(images) @ centroid).mean())

    def cosine_to_prompts(pairs: list[tuple[Image.Image, str]]) -> float:
        images, texts = zip(*pairs)
        image_emb, text_emb = embed_images(list(images)), embed_texts(list(texts))
        return float((image_emb * text_emb).sum(dim=-1).mean())

    # style_gain rides on the off-domain prompts only: the corpus cannot contain a
    # bicycle, so similarity to the corpus there is style and nothing else.
    base_off = [baseline[(p, s)] for p in offdomain for s in seeds]
    base_style = cosine_to_corpus(base_off)
    base_prompt_fit = cosine_to_prompts([(baseline[(p, s)], p) for p in prompts for s in seeds])
    base_saturation = float(np.mean([_saturation(im) for im in baseline.values()]))

    metrics: dict[float, dict[str, float]] = {}
    for w in weights:
        lora_off = [styled[(w, p, s)] for p in offdomain for s in seeds]
        all_lora = [styled[(w, p, s)] for p in prompts for s in seeds]
        metrics[w] = {
            "style_gain": cosine_to_corpus(lora_off) - base_style,
            "prompt_drop": base_prompt_fit
            - cosine_to_prompts([(styled[(w, p, s)], p) for p in prompts for s in seeds]),
            "burn_in": float(np.mean([_saturation(im) for im in all_lora])) / max(base_saturation, 1e-6),
        }

    recommended = _recommend_weight(metrics)
    verdict = _verdict(metrics, recommended)

    rows: list[tuple[str, list[Image.Image]]] = [
        (f"REAL {artist.label} (training data)", reference[: len(prompts)]),
        ("no LoRA (base SD 1.5)", [baseline[(p, seeds[0])] for p in prompts]),
    ]
    for w in weights:
        m = metrics[w]
        rows.append((
            f"LoRA w={w}  style {m['style_gain']:+.3f}  prompt {-m['prompt_drop']:+.3f}  burn {m['burn_in']:.2f}x",
            [styled[(w, p, seeds[0])] for p in prompts],
        ))
    sheet = _sheet(rows)

    for w in weights:
        m = metrics[w]
        print(
            f"  w={w}: style_gain {m['style_gain']:+.3f} | prompt_drop {m['prompt_drop']:+.3f} "
            f"| burn_in {m['burn_in']:.2f}x"
        )
    print(f"  -> {verdict} (best strength: {recommended})")

    del pipe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "artist": artist_id,
        "source": source,
        "metrics": metrics,
        "recommended_weight": recommended,
        "verdict": verdict,
        "passed": verdict == SHIP,  # what export_artist gates the Hub push on
        "sheet": sheet,
        "reference_count": len(reference),
    }

def _recommend_weight(metrics: dict[float, dict[str, float]]) -> float | None:
    """The strongest strength that still follows the prompt and hasn't cooked —
    i.e. the most style you can have without paying for it elsewhere. This is a
    sensible default for the app's Style-strength slider, per LoRA."""
    healthy = [
        w for w, m in metrics.items()
        if m["prompt_drop"] <= SCORECARD_MAX_PROMPT_DROP and m["burn_in"] <= SCORECARD_MAX_BURN_IN
    ]
    return max(healthy) if healthy else None

SHIP = "SHIP"

def _verdict(metrics: dict[float, dict[str, float]], recommended: float | None) -> str:
    best = max(metrics, key=lambda w: metrics[w]["style_gain"])
    if metrics[best]["style_gain"] < SCORECARD_MIN_STYLE_GAIN:
        return "WEAK — barely reads as this artist; retrain (more/cleaner data) or drop it"
    if recommended is None:
        return "OVERCOOKED — style is there but it ignores the prompt / burns in; export an earlier checkpoint"
    return SHIP

def show_scorecard(result: dict) -> None:
    """Display the contact sheet inline in a notebook. The gate must never just say
    "blocked" — the whole point is that you can SEE why (top row = the artist's real
    work). Degrades to a printed path outside Jupyter, so it is safe to call anywhere."""
    print(f"[{result['artist']}] {result['verdict']}  (best strength: {result['recommended_weight']})")
    try:
        from IPython.display import display
    except ImportError:
        return
    display(result["sheet"])

def save_scorecard(result: dict, work_dir: str = "lora_work") -> str:
    out_dir = os.path.join(work_dir, result["artist"], "validation")
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{result['artist']}_scorecard.png")
    result["sheet"].save(path)
    print(f"Saved scorecard -> {path}")
    return path

def scorecard_table(results: list[dict]) -> str:
    """All scored LoRAs as one markdown table, ranked by style gain — the ship /
    fix / retrain decision on a single screen, and a drop-in for the docs page."""
    rows = sorted(
        results, key=lambda r: max(m["style_gain"] for m in r["metrics"].values()), reverse=True
    )
    lines = [
        "| LoRA | style gain | prompt drop | burn-in | best strength | verdict |",
        "|---|---|---|---|---|---|",
    ]
    for r in rows:
        w = r["recommended_weight"] or max(r["metrics"], key=lambda k: r["metrics"][k]["style_gain"])
        m = r["metrics"][w]
        lines.append(
            f"| {r['artist']} | {m['style_gain']:+.3f} | {m['prompt_drop']:+.3f} | "
            f"{m['burn_in']:.2f}x | {r['recommended_weight'] or '—'} | {r['verdict'].split(' — ')[0]} |"
        )
    return "\n".join(lines)


### Step 6 — export + push
_(`training/export.py`)_

Ships the confirmed LoRA: stages a local copy in `app_loras/` and pushes it to its Hugging Face repo (with a model card) under the standard `pytorch_lora_weights.safetensors` filename, so the app's registry can load it by repo id.

In [ ]:
#@title export.py
"""Step 6 of the LoRA pipeline: ship the validated LoRA to a local `app_loras/` copy
and to its Hugging Face repo (what registry.py points at), under the standard
`pytorch_lora_weights.safetensors` filename diffusers expects when loading by repo id,
together with a model card (trigger word, base model, licence/source)."""

def _find_weights(snapshot_dir: str) -> str:
    preferred = os.path.join(snapshot_dir, "pytorch_lora_weights.safetensors")
    if os.path.exists(preferred):
        return preferred
    files = glob.glob(os.path.join(snapshot_dir, "*.safetensors"))
    if not files:
        raise SystemExit(f"No .safetensors weights found in {snapshot_dir}")
    return files[0]

def stage_local(snapshot_dir: str, artist_id: str, app_loras_dir: str = "app_loras") -> str:
    os.makedirs(app_loras_dir, exist_ok=True)
    src = _find_weights(snapshot_dir)
    dst = os.path.join(app_loras_dir, f"{artist_id}-lora.safetensors")
    shutil.copyfile(src, dst)
    print(f"staged {src} -> {dst}")
    return dst

def _model_card(artist_label: str, trigger_word: str, repo_id: str) -> str:
    return f"""---
license: creativeml-openrail-m
base_model: stable-diffusion-v1-5/stable-diffusion-v1-5
tags:
- stable-diffusion
- stable-diffusion-diffusers
- lora
- text-to-image
---

# {artist_label} — SD 1.5 style LoRA

Style LoRA for Stable Diffusion 1.5, produced for the **Latent Studio** capstone
(Creative Coding Advanced, TH Nürnberg) by an automated artist→LoRA pipeline
(download → preprocess → caption → train → validate → export).

- **Trigger word:** `{trigger_word}`
- **Base model:** `stable-diffusion-v1-5/stable-diffusion-v1-5`
- **Training data:** public-domain (CC0) artworks from the Art Institute of
  Chicago Open Access collection, filtered to this artist.

```python
pipe.load_lora_weights("{repo_id}")
image = pipe("{trigger_word}, a small boat crossing a river beneath a mountain").images[0]
```
"""

def push_to_hub(weights_path: str, repo_id: str, artist_label: str, trigger_word: str, private: bool = False) -> str:
    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(repo_id, repo_type="model", exist_ok=True, private=private)
    api.upload_file(
        path_or_fileobj=weights_path,
        path_in_repo="pytorch_lora_weights.safetensors",
        repo_id=repo_id,
    )
    api.upload_file(
        path_or_fileobj=_model_card(artist_label, trigger_word, repo_id).encode("utf-8"),
        path_in_repo="README.md",
        repo_id=repo_id,
    )
    print(f"pushed LoRA -> https://huggingface.co/{repo_id}")
    return repo_id

def export(
    snapshot_dir: str,
    artist_id: str,
    hf_repo: str,
    artist_label: str,
    trigger_word: str,
    app_loras_dir: str = "app_loras",
    push: bool = True,
    private: bool = False,
) -> str:
    """Stages the chosen snapshot locally and (unless push=False) pushes it to
    `hf_repo`. Returns the local staged path."""
    weights_path = stage_local(snapshot_dir, artist_id, app_loras_dir)
    if push:
        try:
            push_to_hub(weights_path, hf_repo, artist_label, trigger_word, private)
        except Exception as exc:
            raise RuntimeError(
                f"Hugging Face push to '{hf_repo}' failed — but the weights are safely staged "
                f"locally at {weights_path}, so no work is lost.\n"
                "This is almost always a token-permission issue: your HF token needs WRITE access. "
                "Create one at https://huggingface.co/settings/tokens (role 'Write'), re-run the "
                "login cell, then just call export_artist(...) again (no need to retrain).\n"
                f"Original error: {exc}"
            ) from exc
    else:
        print("push=False — skipped Hugging Face upload (local staging only).")
    return weights_path


### Orchestrator
_(`training/pipeline.py`)_

Ties the steps together as separate callables (`prepare_data` → `train` → `validate_artist` → `score_artist` → `export_artist`), keeping the validation gate and the scorecard as their own steps so you can inspect the grids before exporting. `revalidate_all` re-runs validation + scorecard over a whole list of already-trained artists (from their Drive checkpoints, no retraining) and writes one combined ranked table to Drive via `save_scorecard_table` — that's the single-device before/after comparison. These are what the sections below call.

In [ ]:
#@title pipeline.py
"""Orchestrator for the LoRA automation pipeline: artist id -> downloaded ->
preprocessed -> captioned -> trained -> validated -> exported LoRA. Exposed as
separate steps (not one monolithic run) so the validation gate can be looked
at before exporting. Per-artist work lives under work_dir/<artist_id>/."""

def _paths(artist_id: str, work_dir: str) -> dict[str, str]:
    base = os.path.join(work_dir, artist_id)
    return {
        "raw": os.path.join(base, "raw"),          # CC0 originals (provenance)
        "dataset": os.path.join(base, "dataset"),  # 512² images + metadata.jsonl
        "output": os.path.join(base, "output"),    # final weights + checkpoint-*
        "validation": os.path.join(base, "validation"),  # saved grids (docs)
        "export": os.path.join(base, "export"),    # staged .safetensors for the app
    }

def prepare_data(artist_id: str, work_dir: str = "lora_work") -> str:
    """Steps 1-3: download -> preprocess -> caption. Returns the dataset dir
    (containing the cleaned 512² images + metadata.jsonl)."""
    artist = ARTISTS[artist_id]
    paths = _paths(artist_id, work_dir)
    # download extra candidates since preprocessing drops some
    download_artist(
        artist.search_name, paths["raw"], artist.artist_match, target=TARGET_IMAGE_COUNT * 2,
        image_width=DOWNLOAD_WIDTH,
    )
    kept = preprocess_folder(
        paths["raw"], paths["dataset"], resolution=RESOLUTION,
        min_size=MIN_IMAGE_SIZE, max_aspect=MAX_ASPECT_RATIO, target=TARGET_IMAGE_COUNT,
    )
    if kept < 10:
        raise SystemExit(f"Only {kept} usable images for '{artist_id}' — too few for a stable LoRA.")
    caption_folder(paths["dataset"], artist.trigger_word)
    return paths["dataset"]

def _training_steps(dataset_dir: str) -> tuple[int, int]:
    """Derive training length from the dataset actually being trained on. Steps *per
    image* is the meaningful knob: a fixed total silently overtrains a small set (19
    images x 2400 steps = ~126 passes per image, which memorises those 19 pictures
    instead of abstracting a style) and undertrains a large one. Returns
    (max_train_steps, checkpointing_steps)."""
    count = len(glob.glob(os.path.join(dataset_dir, "*.png")))
    steps = min(MAX_TRAIN_STEPS, max(MIN_TRAIN_STEPS, STEPS_PER_IMAGE * count))
    checkpointing = max(1, steps // CHECKPOINT_COUNT)
    print(
        f"{count} images -> {steps} steps ({steps / max(count, 1):.0f}/image), "
        f"checkpoint every {checkpointing}"
    )
    return steps, checkpointing

def train(
    artist_id: str,
    dataset_dir: str | None = None,
    work_dir: str = "lora_work",
    diffusers_repo: str = "diffusers",
    resume_from_checkpoint: str | None = None,
) -> str:
    """Step 4: LoRA training. Returns the output dir (final weights + checkpoints).
    Pass `resume_from_checkpoint="latest"` to continue an interrupted run instead
    of restarting (only useful when `work_dir` persists, e.g. on Google Drive)."""
    artist = ARTISTS[artist_id]
    paths = _paths(artist_id, work_dir)
    dataset = dataset_dir or paths["dataset"]
    max_train_steps, checkpointing_steps = _training_steps(dataset)
    run_training(
        dataset_dir=dataset,
        output_dir=paths["output"],
        trigger_word=artist.trigger_word,
        model_name=BASE_MODEL,
        diffusers_repo=diffusers_repo,
        max_train_steps=max_train_steps,
        learning_rate=LEARNING_RATE,
        rank=RANK,
        resolution=RESOLUTION,
        seed=TRAIN_SEED,
        checkpointing_steps=checkpointing_steps,
        resume_from_checkpoint=resume_from_checkpoint,
    )
    return paths["output"]

def validate_artist(artist_id: str, output_dir: str | None = None, work_dir: str = "lora_work") -> dict:
    """Step 5: the validation gate. Returns validate()'s result dict — inspect
    its `checkpoint_grid` / `weight_grid` before exporting."""
    artist = ARTISTS[artist_id]
    paths = _paths(artist_id, work_dir)
    return validate(
        output_dir=output_dir or paths["output"],
        base_model=BASE_MODEL,
        trigger_word=artist.trigger_word,
        content_prompt=artist.validation_prompt,
        seeds=VALIDATION_SEEDS,
        cfg=VALIDATION_CFG,
        steps=VALIDATION_STEPS,
        weights=tuple(VALIDATION_WEIGHTS),
        offdomain_prompts=tuple(VALIDATION_OFFDOMAIN_PROMPTS),
    )

def save_validation_grids(artist_id: str, validation_result: dict, work_dir: str = "lora_work") -> list[str]:
    """Persist the three validation grids as PNGs under work_dir/<artist>/validation/,
    named by artist + the snapshot they were built on."""
    out_dir = _paths(artist_id, work_dir)["validation"]
    os.makedirs(out_dir, exist_ok=True)
    snapshot = validation_result.get("sweep_snapshot") or "final"
    targets = [
        ("checkpoint_grid", f"{artist_id}_checkpoint.png"),
        ("weight_grid", f"{artist_id}_weights_{snapshot}.png"),
        ("generalization_grid", f"{artist_id}_generalization_{snapshot}.png"),
    ]
    saved: list[str] = []
    for key, filename in targets:
        grid = validation_result.get(key)
        if grid is not None:
            path = os.path.join(out_dir, filename)
            grid.save(path)
            saved.append(path)
    print(f"Saved {len(saved)} validation grid(s) to {out_dir}")
    return saved

def recommended_dir(validation_result: dict) -> str | None:
    """The snapshot dir for validate()'s recommended label, once you've eyeballed
    the grids and agree with the recommendation."""
    recommended = validation_result.get("recommended")
    for name, path in validation_result.get("lora_dirs", []):
        if name == recommended:
            return path
    return None

def snapshot_dir(validation_result: dict, step) -> str:
    """Resolve a checkpoint by step number (e.g. 1200, or "final") to its snapshot
    dir, from validation_result["lora_dirs"] — for exporting a specific snapshot
    instead of the recommended one."""
    label = "final" if str(step) == "final" else f"checkpoint-{step}"
    dirs = dict(validation_result.get("lora_dirs", []))
    if label not in dirs:
        available = [name for name, _ in validation_result.get("lora_dirs", [])]
        raise ValueError(f"Snapshot '{label}' not available. Choose a step from: {available}")
    return dirs[label]

def compare_snapshot(artist_id: str, validation_result: dict, step, work_dir: str = "lora_work") -> dict:
    """Re-render the weight-sweep + generalisation grids for ONE checkpoint (by
    step number, or "final") — no retraining. Saves the PNGs to
    work_dir/<artist>/validation/ and returns {"weight_grid",
    "generalization_grid", "snapshot"}."""
    artist = ARTISTS[artist_id]
    label = "final" if str(step) == "final" else f"checkpoint-{step}"
    grids = render_snapshot_grids(
        snapshot_dir=snapshot_dir(validation_result, step),
        base_model=BASE_MODEL,
        trigger_word=artist.trigger_word,
        content_prompt=artist.validation_prompt,
        seeds=VALIDATION_SEEDS,
        cfg=VALIDATION_CFG,
        steps=VALIDATION_STEPS,
        weights=tuple(VALIDATION_WEIGHTS),
        offdomain_prompts=tuple(VALIDATION_OFFDOMAIN_PROMPTS),
    )
    out_dir = _paths(artist_id, work_dir)["validation"]
    os.makedirs(out_dir, exist_ok=True)
    for key, filename in [
        ("weight_grid", f"{artist_id}_weights_{label}.png"),
        ("generalization_grid", f"{artist_id}_generalization_{label}.png"),
    ]:
        if grids.get(key) is not None:
            grids[key].save(os.path.join(out_dir, filename))
    grids["snapshot"] = label
    print(f"Rendered + saved grids for {label} -> {out_dir}")
    return grids

def score_artist(
    artist_id: str,
    lora_source: str | None = None,
    work_dir: str = "lora_work",
    save: bool = True,
    seeds: list[int] | None = None,
) -> dict:
    """Step 5b: grade a LoRA against the artist's real artworks. Pass a snapshot
    dir to grade a candidate, or leave `lora_source` None to grade what's already
    published on the Hub. Returns score_lora()'s dict."""
    extra = {"seeds": seeds} if seeds else {}
    result = score_lora(artist_id, lora_source=lora_source, work_dir=work_dir, **extra)
    if save:
        save_scorecard(result, work_dir)
    return result

def save_scorecard_table(results: list[dict], work_dir: str = "lora_work", tag: str = "") -> str:
    """Write the combined, ranked scorecard table + a copy of every contact sheet
    into work_dir/scorecards/. `tag` (e.g. "before"/"after") keeps two runs from
    overwriting each other. Returns the table's path."""
    out_dir = os.path.join(work_dir, "scorecards")
    os.makedirs(out_dir, exist_ok=True)
    suffix = f"_{tag}" if tag else ""
    table_path = os.path.join(out_dir, f"scorecard{suffix}.md")
    with open(table_path, "w", encoding="utf-8") as f:
        f.write(scorecard_table(results) + "\n")
    for r in results:
        r["sheet"].save(os.path.join(out_dir, f"{r['artist']}_scorecard{suffix}.png"))
    print(f"Wrote {table_path} + {len(results)} contact sheet(s) to {out_dir}")
    return table_path

def revalidate_all(
    artist_ids: list[str],
    work_dir: str = "lora_work",
    validate: bool = True,
    grade_published: bool = False,
    tag: str = "",
) -> list[dict]:
    """Re-run validation + scorecard over already-trained artists (reading
    checkpoints from work_dir/<artist>/output) without retraining — the batched,
    single-device way to produce a before/after comparison across the roster
    (MPS and CUDA don't render the same seed identically, so scores from two
    machines aren't comparable). Grades the local snapshot by default;
    `grade_published=True` grades the Hub LoRA instead. `validate=False` skips
    the slower validation grids. One unreadable artist is skipped, not fatal."""
    results: list[dict] = []
    for artist_id in artist_ids:
        try:
            if validate:
                result = validate_artist(artist_id, work_dir=work_dir)
                save_validation_grids(artist_id, result, work_dir)
            source = None if grade_published else _paths(artist_id, work_dir)["output"]
            results.append(score_artist(artist_id, lora_source=source, work_dir=work_dir))
        except Exception as exc:
            print(f"[{artist_id}] SKIPPED — {exc}")
    if results:
        save_scorecard_table(results, work_dir, tag)
    return results

def export_artist(
    artist_id: str,
    snapshot_dir: str,
    app_loras_dir: str = "app_loras",
    push: bool = True,
    work_dir: str = "lora_work",
    gate: bool = True,
    force: bool = False,
    score: dict | None = None,
) -> str:
    """Step 6: stage the confirmed snapshot locally + push to its HF repo.

    The scorecard gates the push — a LoRA that can't show it resembles the artist
    doesn't get published by accident, but is always staged locally first (a
    refused push loses no work). Pass `score` to reuse an existing scorecard
    instead of regenerating it. Escape hatches: `force=True` pushes despite a
    blocking verdict (after you've looked); `gate=False` skips scoring entirely."""
    artist = ARTISTS[artist_id]

    if push and gate:
        score = score or score_artist(artist_id, lora_source=snapshot_dir, work_dir=work_dir)
        show_scorecard(score)  # always look — this is the visual comparison, not a formality
        if not score["passed"] and not force:
            staged = export(
                snapshot_dir=snapshot_dir, artist_id=artist.id, hf_repo=artist.hf_repo,
                artist_label=artist.label, trigger_word=artist.trigger_word,
                app_loras_dir=app_loras_dir, push=False,
            )
            raise SystemExit(
                f"PUSH BLOCKED — the scorecard says: {score['verdict']}\n"
                f"The weights are staged locally at {staged}, so nothing is lost.\n"
                "Compare the contact sheet above (top row = the artist's REAL work) — also saved to\n"
                f"  {os.path.join(work_dir, artist_id, 'validation', artist_id + '_scorecard.png')}\n"
                "  * OVERCOOKED -> export an earlier snapshot instead, no retraining needed:\n"
                "      export_artist(ARTIST, snapshot_dir(result, 900), ...)\n"
                "  * WEAK -> a data problem; retrain with more/cleaner images, or drop the artist.\n"
                "  * You looked and you disagree -> export_artist(..., force=True)."
            )

    return export(
        snapshot_dir=snapshot_dir,
        artist_id=artist.id,
        hf_repo=artist.hf_repo,
        artist_label=artist.label,
        trigger_word=artist.trigger_word,
        app_loras_dir=app_loras_dir,
        push=push,
    )


## 6. Persistent storage (Google Drive)
Mount Drive so datasets, checkpoints, validation grids and exports survive a runtime disconnect and are reusable across sessions — everything lands under `WORK_DIR/<artist>/{raw,dataset,output,validation,export}`. This is also what makes training **resumable** (below) and keeps the grids around for the documentation. Off Colab it falls back to a local `lora_work/` folder.

In [ ]:
#@title Mount Google Drive
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/latent-studio-training"
except ModuleNotFoundError:
    WORK_DIR = "lora_work"  # not on Colab
os.makedirs(WORK_DIR, exist_ok=True)
print("Work dir:", WORK_DIR)

## 7. Pre-fetch the models (run this before training)
This pipeline downloads from the Hub **three times**, in three different steps: **BLIP** when it captions, **SD 1.5** when it trains, **CLIP** when it scores. Since **2026-07-13** the Hub's CDN intermittently rejects its own signed download links (`403 … SignatureError: invalid key pair id`, [huggingface/datasets#8328](https://github.com/huggingface/datasets/issues/8328)) — a Hugging Face outage, unrelated to this project. Hitting that an hour into a training run costs you the run.

So pull all three **now**, with retries. The cache **resumes**: every attempt keeps whatever it already downloaded, so a retry loop works its way through even when most requests fail. If the cell dies anyway, just run it again — nothing is lost. Afterwards the pipeline reads everything from disk.

In [ ]:
#@title Pre-fetch the models
import time

RETRIES = 40


def fetch(label, call):
    """Retry past the Hub CDN's intermittent 403s. The cache resumes, so every
    attempt makes progress even if most of them fail."""
    for attempt in range(1, RETRIES + 1):
        try:
            return call()
        except Exception as exc:
            flaky = any(s in str(exc) for s in ("403", "SignatureError", "Connection"))
            if not flaky or attempt == RETRIES:
                raise
            print(f"  {label}: Hub CDN hiccup ({attempt}/{RETRIES}), retrying...")
            time.sleep(3)

from diffusers import DiffusionPipeline

print("Fetching SD 1.5 (default weights — training does not use the fp16 variant)...")
fetch("SD 1.5", lambda: DiffusionPipeline.download(BASE_MODEL, use_safetensors=True))

print("Fetching BLIP (captioning, step 3)...")
fetch("BLIP", _load_blip)

print("Fetching CLIP (scorecard, step 5b)...")
fetch("CLIP", _clip)

print("\nAll models cached. Download failures can no longer interrupt a training run.")

## 8. Run the pipeline (one artist per pass)
Pick an artist, then run the cells **in order**. The validation step is a gate: look at the grids before exporting. Repeat for the second artist by setting `ARTIST = "turner"` and re-running this section.

In [ ]:
# Trained: "hokusai", "turner", "monet", "cezanne", "duerer", "hiroshige", "rembrandt"
# Plenty of public-domain data, not trained yet: "lautrec", "renoir", "daumier",
#   "utamaro", "goya", "cassatt", "cameron"
# Too little public-domain data at AIC: "vangogh" (19), "kollwitz" (0 — still in copyright)
ARTIST = "hokusai"
dataset_dir = prepare_data(ARTIST, WORK_DIR)

Train (GPU). Training length **scales with the dataset**: ~40 steps per image, floored at 800 and capped at 6000 — so a 150-image set runs ~6000 steps (roughly 20 min on an L4/G4), while a 20-image set runs 800 instead of memorising itself at 2400. The cell prints the derived step count. **Interrupted mid-run?** Re-run this cell as `train(ARTIST, work_dir=WORK_DIR, resume_from_checkpoint="latest")` to continue from the last checkpoint on Drive. Resume continues the *same* run only — after changing artist / trigger / hyperparameters, train fresh (leave resume off, and clear `WORK_DIR/<artist>/output` if it still holds an old run's checkpoints).

In [ ]:
output_dir = train(ARTIST, work_dir=WORK_DIR)

Validate — inspect all three grids before trusting the recommendation. The weight-sweep + generalization grids are built on the **recommended** snapshot (the one you'll export), not blindly on `final`:

In [ ]:
result = validate_artist(ARTIST, work_dir=WORK_DIR)
result["checkpoint_grid"]

In [ ]:
result["weight_grid"]

In [ ]:
result["generalization_grid"]  # does the style transfer to off-domain prompts?

Persist the three grids to Drive (named by artist + snapshot — doubles as documentation evidence):

In [ ]:
save_validation_grids(ARTIST, result, WORK_DIR)

**(Optional) Re-check a specific checkpoint.** The recommendation is only a starting point. If a different snapshot looks better in the checkpoint grid above, judge it the same way here: set `STEP` to that checkpoint's step **number** (e.g. `1200`, or `"final"`) and run — it re-renders that snapshot's weight sweep + generalization grids (no retraining) and saves them to Drive. `result["lora_dirs"]` lists the available steps.

In [ ]:
STEP = 1200  # checkpoint step to inspect (or "final")
cmp = compare_snapshot(ARTIST, result, STEP, WORK_DIR)
cmp["generalization_grid"]

In [ ]:
cmp["weight_grid"]

### Scorecard — does it actually look like the artist?
The grids above compare the LoRA against the *base model*. They cannot tell you whether it resembles the **artist's real work**, which is the thing you actually trained for. This does: it renders a contact sheet with the real training artworks in the top row and our generations below, and reports **style gain**, **prompt drop** and **burn-in** (see the Step 5b section for what each one means).

Grade the snapshot you're about to ship — or pass nothing to grade whatever is already **published on the Hub**, i.e. exactly what the app will load:

```python
score_artist(ARTIST, work_dir=WORK_DIR)                              # the published LoRA
score_artist(ARTIST, recommended_dir(result), work_dir=WORK_DIR)     # a local snapshot
```

In [ ]:
SNAPSHOT = recommended_dir(result)  # or: snapshot_dir(result, 1200)
score = score_artist(ARTIST, SNAPSHOT, work_dir=WORK_DIR)
print(score["verdict"], "| best strength:", score["recommended_weight"])
score["sheet"]  # top row = the artist's REAL work — compare it against the rows below

### Export the chosen snapshot — the scorecard **gates the push**
Publishing is the step that matters: the Hub copy is what the app loads. So a LoRA that cannot show it resembles the artist **does not get pushed by accident** — `export_artist` refuses, stages the weights locally anyway (no work is lost) and tells you which fix applies:

- **OVERCOOKED** → export an *earlier* snapshot. **No retraining** — the checkpoints are already on Drive: `export_artist(ARTIST, snapshot_dir(result, 900), ...)`.
- **WEAK** → a *data* problem. More/cleaner images, or drop the artist.
- **You looked at the sheet and disagree** → `force=True`. A heuristic must never overrule a human who has actually looked.

Passing `score=score` reuses the scorecard from the cell above instead of generating everything a second time. The contact sheet is displayed either way — blocked or not, you always see what you shipped.

In [ ]:
export_artist(
    ARTIST, SNAPSHOT,
    app_loras_dir=os.path.join(WORK_DIR, ARTIST, "export"),
    work_dir=WORK_DIR,
    score=score,  # reuse the scorecard above; drop this and it re-scores
    # force=True,  # push despite a blocking verdict (only after you've looked)
)

### Re-push a snapshot that is already trained
If an upload died halfway — the Hub repo exists but has no `pytorch_lora_weights.safetensors` — nothing is lost: the weights are on Drive. Push them again **without retraining**. Run the setup cells (install → login → imports → Drive), then this one, pointing it straight at the snapshot folder: `export_artist` takes a plain directory, so no `result` and no training run are needed.

Two ways, pick by whether this snapshot has ever been judged:
- **Never scored** (the usual case) → leave the gate on and run this on a **GPU** runtime. It scores first, shows you the contact sheet, and only then publishes.
- **Already scored and you just want the bytes on the Hub** → `gate=False`. Then no images are generated at all and a **CPU runtime is enough** (costs no GPU quota). Only use this when you have actually looked at the LoRA before.

In [ ]:
# ARTIST = "rembrandt"
# SNAPSHOT = os.path.join(WORK_DIR, ARTIST, "output")  # or ".../output/checkpoint-3600"

# GPU runtime — scores it, shows the sheet, then pushes if it passes:
# export_artist(ARTIST, SNAPSHOT, app_loras_dir=os.path.join(WORK_DIR, ARTIST, "export"), work_dir=WORK_DIR)

# CPU runtime — upload only, no scoring (ONLY for a snapshot you already judged):
# export_artist(ARTIST, SNAPSHOT, app_loras_dir=os.path.join(WORK_DIR, ARTIST, "export"), gate=False)

## 9. Re-validate & score everything already trained (batch, no retraining)
The section above trains and ships **one** artist. This one does the reverse: it takes the artists you have **already** trained — whose checkpoints sit in Drive — and re-runs step 5 (validation) + step 5b (scorecard) over all of them in one pass, **without retraining**. It is the batched form of exactly those two steps, which is why it belongs here between *train* and a per-artist *export*.

**Why it exists — the single-device rule.** MPS (a laptop) and CUDA (this T4) do **not** render the same image for the same seed: different backend kernels, different precision (fp32 vs fp16). So a score measured on one machine cannot be compared against a score from the other, and *everything that goes into the write-up has to come from one device*. Running this on Colab produces that one consistent set. Tag a run `"before"`, retrain the roster, run it again `"after"`, and the two tables in `WORK_DIR/scorecards/` are directly comparable — that is how you answer *"did the hyperparameter fix actually help?"* with a number instead of a feeling. A *"it changed nothing"* is just as publishable as a win.

**Everything lands on Drive:** each artist's grids + contact sheet under `WORK_DIR/<artist>/validation/`, and one combined ranked table + all sheets under `WORK_DIR/scorecards/scorecard_<tag>.md`.

**Source & cost.** It grades the **local Drive snapshot** by default (no Hub download — immune to the CDN outage; for the already-published LoRAs that is the same generation that is live). Pass `grade_published=True` to grade each artist's **published Hub** LoRA instead — exactly what the app loads today. Validation is the expensive part (a checkpoint grid + sweeps per artist); `validate=False` skips it and only scores. The list is ordered by demo importance, so a capped runtime still finishes the ones that matter.

In [ ]:
# Already trained (checkpoints on Drive), ordered by how much each matters for the demo.
ARTISTS_TO_SCORE = ["hiroshige", "hokusai", "turner", "rembrandt", "monet", "cezanne", "duerer"]

before = revalidate_all(ARTISTS_TO_SCORE, work_dir=WORK_DIR, validate=True, tag="before")
print(scorecard_table(before))

After retraining the roster (rerun section 8 per artist), rerun the same batch with `tag="after"` and diff the two tables. Cheaper variants, if you only need the numbers or want to grade what is actually published:

```python
# scores only, no validation grids (much faster):
revalidate_all(ARTISTS_TO_SCORE, work_dir=WORK_DIR, validate=False, tag="after")
# grade the PUBLISHED Hub LoRAs instead of the local snapshots:
revalidate_all(ARTISTS_TO_SCORE, work_dir=WORK_DIR, grade_published=True, tag="published")
```

**Negative examples belong in the write-up too.** The thin-dataset artists whose checkpoints are on Drive but never shipped — **Van Gogh** (19 images), **Cassatt** (old 2400-step run) — are worth scoring precisely *because* they should score badly: a WEAK/OVERCOOKED contact sheet next to a SHIP one is the evidence that the quality gate works, not just an assertion that it does. Add their ids to the list to capture them (they will be skipped cleanly if their Drive folder isn't present).